This notebook explores June 2026 U.S. domestic flight data from the Bureau of Transportation (BTS) in preperation for building a flight delay prediction model.

Load the June 2026 BTS flight data for initial exploration and processing.

In [4]:
import pandas as pd
df = pd.read_csv("../data/T_ONTIME_MARKETING.csv")

##Initial Data Exploration
Preview the first few rows to understand the structure and available features in the dataset.

In [5]:
df.head()

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,ARR_DEL15,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,DISTANCE
0,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,520,818,0.0,0.0,0.0,118.0,569.0
1,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,701,959,0.0,0.0,0.0,118.0,569.0
2,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,820,1112,0.0,0.0,0.0,112.0,569.0
3,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,1043,1334,1.0,0.0,0.0,111.0,569.0
4,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,1607,1908,0.0,0.0,0.0,121.0,569.0


In [6]:
df.shape

(671781, 16)

In [7]:
df.columns

Index(['YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'MKT_UNIQUE_CARRIER', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST',
       'CRS_DEP_TIME', 'CRS_ARR_TIME', 'ARR_DEL15', 'CANCELLED', 'DIVERTED',
       'CRS_ELAPSED_TIME', 'DISTANCE'],
      dtype='str')

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 671781 entries, 0 to 671780
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   YEAR                671781 non-null  int64  
 1   MONTH               671781 non-null  int64  
 2   DAY_OF_MONTH        671781 non-null  int64  
 3   DAY_OF_WEEK         671781 non-null  int64  
 4   FL_DATE             671781 non-null  str    
 5   MKT_UNIQUE_CARRIER  671781 non-null  str    
 6   OP_UNIQUE_CARRIER   671781 non-null  str    
 7   ORIGIN              671781 non-null  str    
 8   DEST                671781 non-null  str    
 9   CRS_DEP_TIME        671781 non-null  int64  
 10  CRS_ARR_TIME        671781 non-null  int64  
 11  ARR_DEL15           657766 non-null  float64
 12  CANCELLED           671781 non-null  float64
 13  DIVERTED            671781 non-null  float64
 14  CRS_ELAPSED_TIME    671778 non-null  float64
 15  DISTANCE            671781 non-null  float64


The dataset contains 671,781 flight records and 16 columns. Most fields are complete, although ARR_DEL15 and CRS_ELAPSED_TIME contain missing values that will require further investigation during preprocessing.

In [9]:
df.isnull().sum()

YEAR                      0
MONTH                     0
DAY_OF_MONTH              0
DAY_OF_WEEK               0
FL_DATE                   0
MKT_UNIQUE_CARRIER        0
OP_UNIQUE_CARRIER         0
ORIGIN                    0
DEST                      0
CRS_DEP_TIME              0
CRS_ARR_TIME              0
ARR_DEL15             14015
CANCELLED                 0
DIVERTED                  0
CRS_ELAPSED_TIME          3
DISTANCE                  0
dtype: int64

In [10]:
df[df["ARR_DEL15"].isnull()][["ARR_DEL15", "CANCELLED", "DIVERTED"]].head(20)

,ARR_DEL15,CANCELLED,DIVERTED
59,NaN,1.0,0.0
412,NaN,0.0,1.0
616,NaN,0.0,1.0
782,NaN,0.0,1.0
1172,NaN,0.0,1.0
1298,NaN,1.0,0.0
1379,NaN,0.0,1.0
1427,NaN,0.0,1.0
1562,NaN,0.0,1.0
1655,NaN,0.0,1.0


In [11]:
df[df["ARR_DEL15"].isnull()][["CANCELLED","DIVERTED"]].value_counts()

CANCELLED  DIVERTED
1.0        0.0         11457
0.0        1.0          2558
Name: count, dtype: int64

##Missing Arrival Delay Values

There are 14,015 missing values in "ARR_DEL15". Further inspection shows that all of these flights were either cancelled or diverted:
-11,457 flights were cancelled
-2,558 flights were diverted

Since every missing "ARR_DEL15" value is associated with a cancelled or diverted flight, the missing values are not random. These flights will need to be handled seperately before building the predictoin model.


In [12]:
df["ARR_DEL15"].value_counts()

ARR_DEL15
0.0    491189
1.0    166577
Name: count, dtype: int64

In [13]:
df["ARR_DEL15"].value_counts(normalize=True) * 100

ARR_DEL15
0.0    74.67534
1.0    25.32466
Name: proportion, dtype: float64

###Target Variable Distribution

Among flgihts with a recorded arrival delay status:

-74.68% arrived less than 15 minutes late.
-25.32% arrived 15 minutes late or more.

The target variable is moderately imbalanced, with substantially more non-delayed flights than delayed flights. This will be considered when evaluating the prediction model

##Data Cleaning

Cancelled and diverted flights are excluded because they do not have a valid "ARR_DEL15" outcome. The prediction model will focus on flights that completed their scheduled trio and have a recorded arrival delay status.

In [14]:
df["ARR_DEL15"].notnull()

0         True
1         True
2         True
3         True
4         True
          ... 
671776    True
671777    True
671778    True
671779    True
671780    True
Name: ARR_DEL15, Length: 671781, dtype: bool

In [15]:
df[df["ARR_DEL15"].notnull()]

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,ARR_DEL15,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,DISTANCE
0,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,520,818,0.0,0.0,0.0,118.0,569.0
1,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,701,959,0.0,0.0,0.0,118.0,569.0
2,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,820,1112,0.0,0.0,0.0,112.0,569.0
3,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,1043,1334,1.0,0.0,0.0,111.0,569.0
4,2026,6,1,1,6/1/2026 12:00:00 AM,AA,AA,ABQ,DFW,1607,1908,0.0,0.0,0.0,121.0,569.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
671776,2026,6,30,2,6/30/2026 12:00:00 AM,WN,WN,VPS,BWI,1430,1750,0.0,0.0,0.0,140.0,819.0
671777,2026,6,30,2,6/30/2026 12:00:00 AM,WN,WN,VPS,DAL,1030,1230,0.0,0.0,0.0,120.0,630.0
671778,2026,6,30,2,6/30/2026 12:00:00 AM,WN,WN,VPS,DAL,1810,2010,0.0,0.0,0.0,120.0,630.0
671779,2026,6,30,2,6/30/2026 12:00:00 AM,WN,WN,VPS,HOU,1710,1900,1.0,0.0,0.0,110.0,527.0


In [16]:
df_clean = df[df["ARR_DEL15"].notnull()]

In [17]:
df_clean.shape

(657766, 16)

In [18]:
df_clean["CRS_ELAPSED_TIME"].isnull().sum()

np.int64(1)

In [19]:
df_clean = df_clean[df_clean["CRS_ELAPSED_TIME"].notnull()]

In [20]:
df_clean.shape

(657765, 16)

In [21]:
df_clean.isnull().sum()

YEAR                  0
MONTH                 0
DAY_OF_MONTH          0
DAY_OF_WEEK           0
FL_DATE               0
MKT_UNIQUE_CARRIER    0
OP_UNIQUE_CARRIER     0
ORIGIN                0
DEST                  0
CRS_DEP_TIME          0
CRS_ARR_TIME          0
ARR_DEL15             0
CANCELLED             0
DIVERTED              0
CRS_ELAPSED_TIME      0
DISTANCE              0
dtype: int64

###Cleaning Summary

Flights without a valid "ARR_DEL15" value were excluded because cancelled and diverted flights do not have a standarc arrival delay outcome. One additional flight with a missing scheduled elapsed time was also removed.

The cleaned dataset contains 657,765 flights with no remaining missing values.

In [22]:
df_clean["CANCELLED"].unique()

array([0.])

In [23]:
df_clean.nunique()

YEAR                     1
MONTH                    1
DAY_OF_MONTH            30
DAY_OF_WEEK              7
FL_DATE                 30
MKT_UNIQUE_CARRIER       8
OP_UNIQUE_CARRIER       18
ORIGIN                 366
DEST                   366
CRS_DEP_TIME          1197
CRS_ARR_TIME          1276
ARR_DEL15                2
CANCELLED                1
DIVERTED                 1
CRS_ELAPSED_TIME       437
DISTANCE              1546
dtype: int64

#Eploratory Data Analysis

Explore patterns in flight delays across different features to better understasnd which factors may be useful for prediction.

In [24]:
df_clean.groupby("DAY_OF_WEEK")["ARR_DEL15"].mean()

DAY_OF_WEEK
1    0.253057
2    0.194692
3    0.227904
4    0.276576
5    0.274688
6    0.246236
7    0.308482
Name: ARR_DEL15, dtype: float64

In [25]:
df_clean[["FL_DATE", "DAY_OF_WEEK"]].drop_duplicates().head(7)

,FL_DATE,DAY_OF_WEEK
0,6/1/2026 12:00:00 AM,1
23029,6/2/2026 12:00:00 AM,2
44061,6/3/2026 12:00:00 AM,3
65513,6/4/2026 12:00:00 AM,4
88616,6/5/2026 12:00:00 AM,5
111798,6/6/2026 12:00:00 AM,6
132306,6/7/2026 12:00:00 AM,7
